# 17 - Gold 6: Delta history, time travel, replay proof, query plan and skew
Run notebooks 12 to 16 first. For the replay proof (TC03), run notebooks 13 to 15 a second time and come back to section 1.

In [0]:
from pyspark.sql import functions as F

BRONZE = "capstone6_telecom.bronze"
SILVER = "capstone6_telecom.silver"
GOLD   = "capstone6_telecom.gold"
cat    = GOLD   # gold tables; silver ones use SILVER

## 1. Replay proof: later MERGEs must insert 0 and update 0 rows

In [0]:
for t in ["fact_network_activity", "fact_status_transitions", "fact_outage_interval", "mart_network_daily"]:
    print(t)
    h = spark.sql(f"DESCRIBE HISTORY {cat}.{t}")
    display(h.select("version", "timestamp", "operation",
                     F.col("operationMetrics")["numTargetRowsInserted"].alias("inserted"),
                     F.col("operationMetrics")["numTargetRowsUpdated"].alias("updated"))
             .orderBy(F.col("version").desc()).limit(3))

## 2. Time travel: what did the CDC change in silver_network_current?
The first MERGE seeded every event with its original status. The next MERGE applied the status changes.

In [0]:
hist = spark.sql(f"DESCRIBE HISTORY {SILVER}.silver_network_current")
display(hist.select("version", "timestamp", "operation").orderBy("version"))

v_before = hist.filter("operation = 'MERGE'").orderBy("version").first()["version"]
print("Version before the CDC was applied:", v_before)

spark.sql(f"CREATE OR REPLACE TEMP VIEW _cur_before AS SELECT * FROM {SILVER}.silver_network_current VERSION AS OF {v_before}")
spark.sql(f"CREATE OR REPLACE TEMP VIEW _cur_now AS SELECT * FROM {SILVER}.silver_network_current")

In [0]:
# expect 478 changed statuses (520 changes minus 42 unchanged COMPLETED to COMPLETED)
display(spark.sql("""
SELECT COUNT(*) AS statuses_changed_by_cdc
FROM _cur_before a JOIN _cur_now b USING (measurement_id)
WHERE a.current_status <> b.current_status"""))

display(spark.sql("""
SELECT a.current_status AS status_before, b.current_status AS status_now, COUNT(*) AS rows
FROM _cur_before a JOIN _cur_now b USING (measurement_id)
WHERE a.current_status <> b.current_status
GROUP BY 1, 2 ORDER BY rows DESC"""))

## 3. Query plan of the KPI query (save this as the baseline for the Day 4 OPTIMIZE comparison)

In [0]:
plan = spark.sql(f"""EXPLAIN FORMATTED
SELECT region_id, event_date, SUM(outage_minutes) AS outage_minutes
FROM {cat}.fact_network_activity GROUP BY region_id, event_date""").collect()[0][0]
print(plan)

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {cat}.fact_network_activity"))

## 4. Skew check (a ratio near 1 means evenly spread; much higher means hot keys)

In [0]:
for col in ["region_id", "tower_id", "event_date"]:
    print("Skew by", col)
    display(spark.sql(f"""
    SELECT COUNT(*) AS distinct_values, MAX(c) AS max_rows, ROUND(AVG(c), 1) AS avg_rows,
           ROUND(MAX(c) / AVG(c), 2) AS skew_ratio
    FROM (SELECT {col}, COUNT(*) AS c FROM {cat}.fact_network_activity GROUP BY {col})"""))